In [ ]:
import os
import pickle
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve, average_precision_score, 
    brier_score_loss, auc
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

print("=========================================")
print("  E-COMMERCE MODEL COMPARISON & CALIBRATION")
print("=========================================\n")

# Ensure artifacts directory exists
os.makedirs('../../backend/artifacts', exist_ok=True)

# 1. Load the Data
train_df = pd.read_csv('../../data/processed/train.csv')
test_df = pd.read_csv('../../data/processed/test.csv')

X_train = train_df.drop('Revenue', axis=1)
y_train = train_df['Revenue']
X_test = test_df.drop('Revenue', axis=1)
y_test = test_df['Revenue']

# 2. Load the Trained Models for Comparison
print("Loading trained models for comparison...\n")
models = {}
model_files = [
    'logistic_regression.pkl', 
    'decision_tree.pkl', 
    'random_forest.pkl', 
    'xgboost.pkl'
]

for filename in model_files:
    filepath = f"../../backend/models/{filename}"
    if os.path.exists(filepath):
        with open(filepath, 'rb') as f:
            # Format name for display (e.g., 'xgboost.pkl' -> 'XGBoost')
            display_name = filename.replace('.pkl', '').replace('_', ' ').title()
            if display_name == 'Xgboost': display_name = 'XGBoost'
            models[display_name] = pickle.load(f)

# 3. Precision-Recall Curve (PR-AUC)
# Crucial for imbalanced datasets (84/16 split)
fig, ax = plt.subplots(figsize=(10, 8))
print("--- Calculating Precision-Recall AUC ---")

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    
    print(f"{name} PR-AUC: {pr_auc:.4f}")
    ax.plot(recall, precision, lw=2, label=f"{name} (PR-AUC = {pr_auc:.3f})")

# Baseline represents the random guessing rate
baseline = y_test.mean()
ax.plot([0, 1], [baseline, baseline], color='navy', lw=2, linestyle='--', label=f'Baseline ({baseline:.3f})')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set